# hipinfer on Colab

LLM inference engine written from scratch in HIP (FlashAttention-style attention, tensor-core GEMM,
fused GEMV/RMSNorm/RoPE kernels), built here with `nvcc` for the attached NVIDIA GPU.

**Runtime → Change runtime type → a GPU.** An **A100** is the most useful: it fits every model next
to its HF reference copy and has the bandwidth to show launch overheads. L4 and T4 work too, with fewer
models. The CPU and TPU runtimes cannot run it: the kernels are GPU code.

Run all cells. The notebook sizes itself to the GPU:

| GPU memory | models | engine comparison (llama.cpp, vLLM, torch.compile, HF) |
|---|---|---|
| 40 GB+ (A100) | all 7, up to Llama-3.1-8B | on, 5 models |
| 22–39 GB (L4) | up to Llama-3.2-3B | on, up to 3B |
| less (T4) | 4 small models | off |

Expected time on an A100: about 2–2.5 h in all, of which about 1 h is the engine comparison. Each model
is deleted from disk once it is done, so about 60 GB of free disk is enough. It ends by downloading
`hipinfer_results_<gpu>.zip`; send that back.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| capability", torch.cuda.get_device_capability())

Unpack the sources (embedded below so no clone is needed).

In [ ]:
import base64, io, tarfile, os
BLOB = "..."
os.makedirs("/content/hipinfer", exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(BLOB)), mode="r:gz").extractall("/content/hipinfer")
%cd /content/hipinfer
!ls -R src scripts tests

**Settings.** Chosen from the GPU's memory; edit `MODELS`, `RUN_BASELINES` or `BASELINE_MODELS` to
override. Weights are exported from the HF checkpoints on first use (`unsloth/Llama-3.2-*` and
`unsloth/Meta-Llama-3.1-8B` are ungated mirrors of Meta's weights). Mistral-7B is gated: accept its terms
on huggingface.co and add an `HF_TOKEN` Colab secret (key icon, left sidebar) to include it; without that it
is skipped.

In [ ]:
!pip install -q datasets nvidia-ml-py
import os, shutil, subprocess, torch
gpu = torch.cuda.get_device_name().replace(" ", "_")
mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
cap = "%d%d" % torch.cuda.get_device_capability()
if mem_gb >= 38:
    MODELS = "tinyllama llama32-1b qwen25-0.5b qwen25-1.5b llama32-3b mistral-7b llama31-8b"
elif mem_gb >= 22:
    MODELS = "tinyllama llama32-1b qwen25-0.5b qwen25-1.5b llama32-3b"
else:
    MODELS = "tinyllama llama32-1b qwen25-0.5b qwen25-1.5b"
try:  # the HF_TOKEN Colab secret, exported so the scripts' own processes see it too
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("no HF_TOKEN Colab secret (or notebook access to it is off)")
try:  # Mistral's checkpoint is gated: needs its terms accepted on HF and an HF_TOKEN Colab secret
    from huggingface_hub import auth_check
    auth_check("mistralai/Mistral-7B-v0.3")
except ImportError:
    pass
except Exception:
    print("no access to mistralai/Mistral-7B-v0.3 (gated) -> skipping mistral-7b")
    MODELS = MODELS.replace(" mistral-7b", "")
RUN_BASELINES = mem_gb >= 22
BASELINE_MODELS = [m for m in ["tinyllama", "llama32-1b", "qwen25-1.5b", "llama32-3b", "llama31-8b"] if m in MODELS.split()]
print(f"{gpu}: {mem_gb:.0f} GB, sm_{cap}, {shutil.disk_usage('/content').free / 1e9:.0f} GB free disk")
print("hipinfer suite:", MODELS)
print("engine comparison:", BASELINE_MODELS if RUN_BASELINES else "off")

**Build and kernel tests**, once: build for this GPU, then every kernel against PyTorch. A background
`nvidia-smi` log records temperature and clocks every 5 s for the whole run
(`results/colab_<gpu>/gpu_log.csv`), so the results can be checked for thermal throttling.

In [ ]:
os.makedirs(f"results/colab_{gpu}", exist_ok=True)
# older drivers call the throttle fields clocks_throttle_reasons.*
why = "clocks_event_reasons" if subprocess.run("nvidia-smi --query-gpu=clocks_event_reasons.active --format=csv",
                                               shell=True, capture_output=True).returncode == 0 else "clocks_throttle_reasons"
gpulog = subprocess.Popen(
    "nvidia-smi --query-gpu=timestamp,temperature.gpu,power.draw,clocks.sm,utilization.gpu,"
    f"{why}.sw_thermal_slowdown,{why}.hw_thermal_slowdown "
    f"--format=csv -l 5 > results/colab_{gpu}/gpu_log.csv", shell=True)
!chmod +x scripts/run_all.sh scripts/run_baselines.sh
!PY=python SKIP_SINGLE=1 SKIP_TP=1 scripts/run_all.sh colab_{gpu}

**Engine comparison setup** (skipped when `RUN_BASELINES` is off). vLLM goes in its own virtual
environment (about 5 min), because it pins its own PyTorch and would otherwise replace the one hipinfer is
built against. llama.cpp is built for this GPU only (about 5 min) on its first use below.

In [ ]:
sh = get_ipython().system
if RUN_BASELINES:
    # llama.cpp's converter needs only these (its requirements file would replace Colab's PyTorch)
    sh("pip install -q --no-cache-dir sentencepiece protobuf ninja")
    sh("[ -x /content/vllm_env/bin/python ] || (python -m venv /content/vllm_env && /content/vllm_env/bin/pip install -q --no-cache-dir vllm)")
    sh("/content/vllm_env/bin/python -c 'import vllm; print(\"vllm\", vllm.__version__)'")
sh("pip cache purge -q 2>/dev/null; df -h /content | tail -1")

**Per model**, one at a time: HF token gate, WikiText-2 perplexity vs HF and the benchmarks (`GEMM_SWEEP`
also times every GEMM tile config on the first model), then the engine comparison for the models in
`BASELINE_MODELS` (llama.cpp, vLLM, torch.compile and HF, with hipinfer re-run beside them). Afterwards the
model's HF download, hipinfer weights and GGUF are deleted, so the disk only ever holds one model (the 8B
peaks at about 50 GB). A model already finished in this runtime is skipped, so the cell can be re-run
after an interruption.

In [ ]:
from huggingface_hub import constants
HF = {"tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "llama32-1b": "unsloth/Llama-3.2-1B",
      "qwen25-0.5b": "Qwen/Qwen2.5-0.5B", "qwen25-1.5b": "Qwen/Qwen2.5-1.5B", "llama32-3b": "unsloth/Llama-3.2-3B",
      "mistral-7b": "mistralai/Mistral-7B-v0.3", "llama31-8b": "unsloth/Meta-Llama-3.1-8B"}
GB = {"tinyllama": 2.2, "llama32-1b": 2.5, "qwen25-0.5b": 1.0, "qwen25-1.5b": 3.1, "llama32-3b": 6.4,
      "mistral-7b": 14.5, "llama31-8b": 16.1}
for i, m in enumerate(MODELS.split()):
    out = f"results/colab_{gpu}/{m}"
    if os.path.exists(f"{out}/bench.json") and (m not in BASELINE_MODELS or not RUN_BASELINES
                                               or os.path.exists(f"results/baselines/colab_{gpu}/{m}/llamacpp.json")):
        print(f"== {m}: done earlier in this runtime, skipped")
        continue
    free = shutil.disk_usage("/content").free / 1e9
    if free < GB[m] * 3 + 5:
        print(f"== {m}: needs about {GB[m] * 3 + 5:.0f} GB of disk, {free:.0f} GB free -> skipped")
        continue
    print(f"######## {m} ({free:.0f} GB free disk)")
    sh(f"PY=python SKIP_BUILD=1 SKIP_KERNEL_TESTS=1 SKIP_TP=1 GEMM_SWEEP={int(i == 0)} PPL_WINDOWS=8 "
       f"scripts/run_all.sh colab_{gpu} {m}")
    gguf = f"/content/{m}-f16.gguf"
    if RUN_BASELINES and m in BASELINE_MODELS:
        sh(f"PY=python LLAMA_CPP=/content/llama.cpp CUDA_ARCHS={cap} GGUF={gguf} MODEL={HF[m]} WEIGHTS={m}.bin "
           f"VLLM_PY=/content/vllm_env/bin/python SCRATCH=/content/tmp scripts/run_baselines.sh colab_{gpu}/{m}")
    # free the disk for the next model: HF download, hipinfer weights, GGUF
    shutil.rmtree(os.path.join(constants.HF_HUB_CACHE, "models--" + HF[m].replace("/", "--")), ignore_errors=True)
    for f in (f"{m}.bin", gguf):
        if os.path.exists(f):
            os.remove(f)

Summary tables (the same script produces results/SUMMARY.md in the repo), then download the results.

In [ ]:
gpulog.terminate()
!python scripts/summarize.py results

In [ ]:
!zip -qr hipinfer_results_{gpu}.zip results
from google.colab import files
files.download(f"hipinfer_results_{gpu}.zip")